In [ ]:
!pip install faster-whisper

In [ ]:
import os
from faster_whisper import WhisperModel
from tqdm import tqdm

# SRT用の時間フォーマット（00:00:00,000）に変換する関数
def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    milliseconds = int((seconds % 1) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{milliseconds:03d}"

# 1. モデルの読み込み
model = WhisperModel("large-v3", device="cuda", compute_type="float16")

# 2. 文字起こしの実行
audio_path = "/content/drive/MyDrive/temp/20260522_ トラン会.mp3"  # 対象ファイルのパス
file_name_with_ext = os.path.basename(audio_path)
file_name, _ = os.path.splitext(file_name_with_ext)

segments, info = model.transcribe(audio_path, beam_size=5, language="ja", word_timestamps=True)

# 音声の総再生時間（秒）を取得
total_duration = info.duration
print(f"🎵 音声ファイルの長さ: {format_time(total_duration)}")

# 3. SRTファイルへの書き込み（進捗表示付き）
output_srt_path = f"./{file_name}.srt"

# tqdmで進捗バーを表示（処理した音声の秒数ベースで進捗を％表示します）
with open(output_srt_path, "w", encoding="utf-8") as f:
    with tqdm(total=total_duration, unit="sec", desc="文字起こし進捗") as pbar:
        last_position = 0

        for i, segment in enumerate(segments, start=1):
            start_time = format_time(segment.start)
            end_time = format_time(segment.end)

            # SRTの標準フォーマットで書き込み
            f.write(f"{i}\n")
            f.write(f"{start_time} --> {end_time}\n")
            f.write(f"{segment.text.strip()}\n\n")

            # 進捗バーの更新（現在のセグメントの終了位置まで進める）
            pbar.update(segment.end - last_position)
            last_position = segment.end

        # 最後に進捗を100%にする
        pbar.update(total_duration - last_position)

print(f"\n✨ SRTファイルの作成が完了しました: {output_srt_path}")

🎵 音声ファイルの長さ: 01:29:11,146


文字起こし進捗:  17%|█▋        | 907.44/5351.1466875 [01:28<07:14, 10.22sec/s]


KeyboardInterrupt: 